# AssetGuard-X raw archive EDA and processing gate

This notebook is read-only with respect to raw Parquet data. It uses **Polars** to inspect the consolidated calendar-month files on Google Drive, including manifest checksums, contract coverage, topics, and duplicate log IDs. It answers these questions before processing starts:

1. Are all expected UTC calendar-month partitions present and correctly bounded?
2. Do the partitions have the required columns and timestamps?
3. Did every configured protocol contract/entity produce data?
4. Are hourly anchors, RPC state rows, duplicates, and partial rows valid?
5. Are the consolidated raw logs complete and traceable, and which event topics are still unknown?

The notebook does not call BigQuery or the Ethereum RPC endpoint. It tests the current ABI decoder against previously unknown archived rows without modifying frozen raw files. A zero event count means only that the selected archive contains no observed row; it is not proof of complete on-chain coverage. Passing this raw-data gate does **not** validate reconstructed prices, labels, graph exposures, or forecasts.

In [ ]:
from __future__ import annotations

import json
import hashlib
import os
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq
from eth_utils import keccak
import yaml
from IPython.display import display

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
NOTEBOOK_ROOT = Path.cwd()
while not (NOTEBOOK_ROOT / 'config' / 'extraction.yaml').exists() and NOTEBOOK_ROOT != NOTEBOOK_ROOT.parent:
    NOTEBOOK_ROOT = NOTEBOOK_ROOT.parent
if not (NOTEBOOK_ROOT / 'config' / 'extraction.yaml').exists():
    raise FileNotFoundError('Open this notebook from the raw-v2 package or start Jupyter below its folder.')
CFG = yaml.safe_load((NOTEBOOK_ROOT / 'config' / 'extraction.yaml').read_text(encoding='utf-8'))
REGISTRY = yaml.safe_load((NOTEBOOK_ROOT / 'config' / 'contracts.yaml').read_text(encoding='utf-8'))
ABI_CONFIG = yaml.safe_load((NOTEBOOK_ROOT / 'config' / 'raw_log_abis.yaml').read_text(encoding='utf-8'))
sys.path.insert(0, str(NOTEBOOK_ROOT / 'src'))
from decode_raw_logs import decode_log_record, make_contract_specs
CONTRACT_SPECS = make_contract_specs(REGISTRY, ABI_CONFIG)
DRIVE_ROOT = Path(os.environ.get('ASSETGUARDX_DRIVE_ROOT', r'G:\My Drive'))
ARCHIVE_ROOT = DRIVE_ROOT / CFG['storage']['project_subdir']
print('Drive root:', DRIVE_ROOT)
print('Archive root:', ARCHIVE_ROOT)
print('Archive exists:', ARCHIVE_ROOT.exists())
if not ARCHIVE_ROOT.exists():
    raise FileNotFoundError(f'Raw archive not found: {ARCHIVE_ROOT}. Set ASSETGUARDX_DRIVE_ROOT to the mounted My Drive folder.')

In [ ]:
# Scope and expected monthly partitions
def parse_utc(value):
    return datetime.fromisoformat(value.replace('Z', '+00:00')).astimezone(timezone.utc)

def month_windows(start, end):
    current = datetime(start.year, start.month, 1, tzinfo=timezone.utc)
    while current < end:
        nxt = datetime(current.year + (current.month == 12), 1 if current.month == 12 else current.month + 1, 1, tzinfo=timezone.utc)
        yield current, min(nxt, end)
        current = nxt

START = parse_utc(CFG['study_window']['start'])
END = parse_utc(CFG['study_window']['validation_end_exclusive'])
MONTHS = [(start.strftime('%Y-%m'), start, end) for start, end in month_windows(START, END)]
DATASETS = ['protocol_events', 'token_activity_events', 'hourly_blocks', 'rpc_hourly']
print('Study start:', START)
print('Validation end:', END)
print('Expected months:', len(MONTHS), [m[0] for m in MONTHS])
print('Target assets:', [k for k, v in REGISTRY['assets'].items() if v.get('target_asset')])

In [ ]:
# Monthly file inventory: rows, schema, time range, and duplicate checks
TIME_COLUMN = {'protocol_events': 'block_timestamp', 'token_activity_events': 'block_timestamp', 'hourly_blocks': 'hour', 'rpc_hourly': 'hour'}
REQUIRED_COLUMNS = {
    'protocol_events': {'contract_key', 'contract_address', 'event_signature', 'block_number', 'block_timestamp', 'transaction_hash', 'args_json'},
    'token_activity_events': {'asset', 'token_address', 'activity_type', 'quantity_raw', 'token_decimals', 'block_number', 'block_timestamp', 'transaction_hash'},
    'hourly_blocks': {'hour', 'block_number', 'block_timestamp'},
    'rpc_hourly': {'hour', 'entity_key', 'state_kind', 'quality_status', 'values_json'},
}

def files_for(dataset, month):
    return sorted((ARCHIVE_ROOT / 'raw' / dataset / f'year={month[:4]}' / f'month={month[5:7]}').glob('*.parquet'))

def inspect_partition(dataset, month, start, end):
    files = files_for(dataset, month)
    base = {'dataset': dataset, 'month': month, 'file_count': len(files), 'rows': 0, 'status': 'missing', 'columns': [], 'missing_columns': [], 'min_time': None, 'max_time': None, 'outside_month_rows': None, 'duplicates': None}
    if not files:
        return base
    try:
        lf = pl.scan_parquet([str(p) for p in files], glob=False)
        columns = set(lf.collect_schema().names())
        base['columns'] = sorted(columns)
        base['missing_columns'] = sorted(REQUIRED_COLUMNS[dataset] - columns)
        base['rows'] = int(lf.select(pl.len()).collect().item())
        time_col = TIME_COLUMN[dataset]
        if time_col in columns:
            stat = lf.select(pl.col(time_col).min().alias('min'), pl.col(time_col).max().alias('max')).collect().row(0)
            base['min_time'], base['max_time'] = [str(v) if v is not None else None for v in stat]
            base['outside_month_rows'] = int(lf.filter(pl.col(time_col).is_null() | (pl.col(time_col) < start) | (pl.col(time_col) >= end)).select(pl.len()).collect().item())
        keys = {'protocol_events': ['transaction_hash', 'log_index', 'contract_address'], 'token_activity_events': ['transaction_hash', 'log_index', 'token_address'], 'hourly_blocks': ['hour'], 'rpc_hourly': ['hour', 'entity_key']}[dataset]
        if set(keys).issubset(columns):
            indexed = lf.filter(pl.col('log_index').is_not_null()) if 'log_index' in keys else lf
            total = int(indexed.select(pl.len()).collect().item())
            unique = int(indexed.select(pl.struct(keys).n_unique()).collect().item())
            base['duplicates'] = total - unique
        expected_hours = int((end - start).total_seconds() // 3600)
        if base['missing_columns']:
            base['status'] = 'schema_error'
        elif base['outside_month_rows']:
            base['status'] = 'outside_month'
        elif dataset == 'hourly_blocks' and base['rows'] != expected_hours:
            base['status'] = 'hour_gap_or_duplicate'
        elif base['duplicates']:
            base['status'] = 'duplicates'
        elif dataset == 'rpc_hourly' and base['rows'] == 0:
            base['status'] = 'empty'
        else:
            base['status'] = 'ok'
    except Exception as exc:
        base['status'] = 'unreadable'
        base['error'] = f'{type(exc).__name__}: {exc}'
    return base

inventory = pl.DataFrame([inspect_partition(ds, month, start, end) for ds in DATASETS for month, start, end in MONTHS])
display(inventory.select(['month', 'dataset', 'status', 'rows', 'file_count', 'min_time', 'max_time', 'outside_month_rows', 'duplicates']))

In [ ]:
# Protocol event coverage: every configured contract should have rows.
protocol_files = sorted((ARCHIVE_ROOT / 'raw' / 'protocol_events').glob('year=*/month=*/*.parquet'))
if protocol_files:
    protocol_events = pl.scan_parquet([str(p) for p in protocol_files], glob=False).select(['contract_key', 'event_signature']).collect()
    contract_counts = protocol_events.group_by('contract_key').len(name='rows')
    signature_counts = protocol_events.group_by(['contract_key', 'event_signature']).len(name='rows')
    configured = []
    for key, meta in REGISTRY['contracts'].items():
        signatures = meta.get('decoded_event_signatures') or []
        if not signatures:
            continue
        n = contract_counts.filter(pl.col('contract_key') == key)['rows'].sum()
        configured.append({'contract_key': key, 'protocol': meta.get('protocol'), 'rows': int(n or 0), 'expected_signatures': len(signatures), 'observed_signatures': int(signature_counts.filter(pl.col('contract_key') == key).height)})
    event_coverage = pl.DataFrame(configured).sort(['protocol', 'contract_key'])
    display(event_coverage)
    signature_rows = []
    for key, meta in REGISTRY['contracts'].items():
        for signature in meta.get('decoded_event_signatures') or []:
            observed = signature_counts.filter((pl.col('contract_key') == key) & (pl.col('event_signature') == signature))['rows'].sum()
            signature_rows.append({'contract_key': key, 'event_signature': signature, 'rows': int(observed or 0)})
    event_signature_coverage = pl.DataFrame(signature_rows).sort(['contract_key', 'event_signature'])
    display(event_signature_coverage)
    missing_event_signatures = event_signature_coverage.filter(pl.col('rows') == 0)
    zero_event_contracts = event_coverage.filter(pl.col('rows') == 0)
    print('Zero-row configured contracts:', zero_event_contracts['contract_key'].to_list())
    print('Configured event signatures with zero observed rows:', missing_event_signatures.height)
else:
    protocol_events = pl.DataFrame()
    event_coverage = pl.DataFrame()
    event_signature_coverage = pl.DataFrame()
    missing_event_signatures = pl.DataFrame()
    zero_event_contracts = pl.DataFrame()
    print('No protocol_events files found')

In [ ]:
# Token activity coverage and RPC quality
token_files = sorted((ARCHIVE_ROOT / 'raw' / 'token_activity_events').glob('year=*/month=*/*.parquet'))
if token_files:
    token_activity = pl.scan_parquet([str(p) for p in token_files], glob=False).select(['asset', 'activity_type']).collect()
    token_coverage = token_activity.group_by(['asset', 'activity_type']).len(name='rows').sort(['asset', 'activity_type'])
    expected_token_assets = {key for key, meta in REGISTRY['assets'].items() if meta.get('target_asset') and key in REGISTRY.get('token_activity', {})}
    observed_token_assets = set(token_activity['asset'].unique().to_list())
    missing_token_assets = sorted(expected_token_assets - observed_token_assets)
    display(token_coverage)
else:
    token_activity = pl.DataFrame()
    token_coverage = pl.DataFrame()
    missing_token_assets = sorted({key for key, meta in REGISTRY['assets'].items() if meta.get('target_asset') and key in REGISTRY.get('token_activity', {})})
    print('No token_activity_events files found')

rpc_files = sorted((ARCHIVE_ROOT / 'raw' / 'rpc_hourly').glob('year=*/month=*/*.parquet'))
if rpc_files:
    rpc = pl.scan_parquet([str(p) for p in rpc_files], glob=False).select(['entity_key', 'state_kind', 'quality_status', 'errors_json']).collect()
    rpc_coverage = rpc.group_by(['entity_key', 'state_kind']).len(name='rows').sort('entity_key')
    display(rpc_coverage)
    rpc_quality = rpc.group_by(['entity_key', 'state_kind', 'quality_status']).len(name='rows').sort(['entity_key', 'quality_status'])
    display(rpc_quality)
    rpc_error_examples = (rpc.filter(pl.col('quality_status') != 'ok')
        .group_by(['entity_key', 'state_kind'])
        .agg([pl.len().alias('partial_rows'), pl.col('errors_json').drop_nulls().first().alias('example_errors')])
        .sort('entity_key'))
    display(rpc_error_examples)
    partial_rpc_rows = rpc.filter(pl.col('quality_status') != 'ok').height
    configured_state_entities = []
    for key, meta in REGISTRY['contracts'].items():
        if not meta.get('state_kind'):
            continue
        if meta.get('state_kind') == 'aave_v2_data_provider':
            configured_state_entities.extend(f"aave_v2_{reserve['symbol']}" for reserve in meta.get('reserves', []))
        else:
            configured_state_entities.append(key)
    observed_state_entities = set(rpc_coverage['entity_key'].to_list())
    zero_rpc_entities = [key for key in configured_state_entities if key not in observed_state_entities]
    print('Partial RPC rows:', partial_rpc_rows)
    print('Zero-row RPC entities:', zero_rpc_entities)
else:
    rpc = pl.DataFrame()
    rpc_coverage = pl.DataFrame()
    rpc_quality = pl.DataFrame()
    rpc_error_examples = pl.DataFrame()
    partial_rpc_rows = 0
    zero_rpc_entities = [key for key, meta in REGISTRY['contracts'].items() if meta.get('state_kind')]
    print('No rpc_hourly files found')

In [ ]:
# Audit the raw-log source rows inside the unified monthly protocol_events files.
RAW_LOG_DATASET = "protocol_events"
RAW_LOG_WINDOW_DAYS = "calendar_month_utc"
RAW_LOG_KEYS = list(CFG["bigquery"].get("raw_log_contracts", []))
RAW_LOG_REQUIRED = {
    "block_number", "block_timestamp", "transaction_hash", "log_index",
    "contract_address", "contract_key", "topic0", "topics", "data",
    "source_event_fingerprint", "source_table",
}
RAW_LOG_DIR = ARCHIVE_ROOT / "raw" / RAW_LOG_DATASET
RAW_LOG_MANIFEST = ARCHIVE_ROOT / "metadata" / "RAW_FROZEN.json"

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def raw_log_windows(start, end, days):
    yield from month_windows(start, end)

manifest_by_name = {}
manifest_parse_errors = []
if RAW_LOG_MANIFEST.exists():
    try:
        frozen_manifest = json.loads(RAW_LOG_MANIFEST.read_text(encoding="utf-8"))
        for record in frozen_manifest.get("files", []):
            if record.get("dataset") == RAW_LOG_DATASET:
                manifest_by_name.setdefault(Path(record.get("path", "")).name, []).append(record)
    except Exception as exc:
        manifest_parse_errors.append({"error": f"{type(exc).__name__}: {exc}"})
else:
    manifest_parse_errors.append({"line": None, "error": f"manifest not found: {RAW_LOG_MANIFEST}"})

raw_window_rows = []
raw_scan_files = []
for window_start, window_end in raw_log_windows(START, END, RAW_LOG_WINDOW_DAYS):
    filename = f"{RAW_LOG_DATASET}_{window_start.strftime('%Y-%m')}.parquet"
    path = RAW_LOG_DIR / f"year={window_start.year}" / f"month={window_start.month:02d}" / filename
    records = manifest_by_name.get(filename, [])
    exists = path.is_file()
    row_count = None
    file_bytes = path.stat().st_size if exists else None
    missing_columns = sorted(RAW_LOG_REQUIRED)
    checksum_checked = False
    row_count_ok = size_ok = window_ok = checksum_ok = metadata_ok = None
    error = None

    if exists:
        try:
            parquet = pq.ParquetFile(path)
            try:
                row_count = int(parquet.metadata.num_rows)
                missing_columns = sorted(RAW_LOG_REQUIRED - set(parquet.schema_arrow.names))
                metadata = parquet.schema_arrow.metadata or {}
                metadata_ok = (metadata.get(b'assetguardx_consolidation_version') == b'assetguardx-protocol-events/v2' and metadata.get(b'assetguardx_partition_by') == b'block_timestamp:calendar_month_utc')
            finally:
                parquet.close()
            if row_count > 0 and not missing_columns:
                raw_scan_files.append(str(path))
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"

    if len(records) == 1:
        record = records[0]
        try:
            window_ok = Path(record["path"]) == path.relative_to(ARCHIVE_ROOT)
            row_count_ok = exists and row_count == int(record["rows"])
            size_ok = exists and file_bytes == int(record["bytes"])
        except (KeyError, TypeError, ValueError):
            window_ok = row_count_ok = size_ok = False
        if exists and record.get("sha256"):
            checksum_checked = True
            try:
                checksum_ok = sha256_file(path) == str(record["sha256"]).lower()
            except Exception as exc:
                checksum_ok = False
                error = error or f"checksum {type(exc).__name__}: {exc}"

    checksum_matches = checksum_ok is True
    checks_ok = (
        exists and len(records) == 1 and window_ok is True and metadata_ok is True
        and row_count_ok is True and size_ok is True and checksum_matches
        and (not missing_columns or row_count == 0) and error is None
    )
    if not exists:
        status = "missing_file"
    elif not records:
        status = "missing_manifest"
    elif len(records) > 1:
        status = "duplicate_manifest_entries"
    elif not checks_ok:
        status = "integrity_or_schema_error"
    elif row_count == 0:
        status = "empty_window_review"
    else:
        status = "ok"

    raw_window_rows.append({
        "window_start": window_start.isoformat(),
        "window_end_exclusive": window_end.isoformat(),
        "file": filename,
        "status": status,
        "rows": row_count,
        "bytes": file_bytes,
        "manifest_entries": len(records),
        "row_count_ok": row_count_ok,
        "size_ok": size_ok,
        "window_ok": window_ok,
        "metadata_ok": metadata_ok,
        "checksum_checked": checksum_checked,
        "checksum_ok": checksum_ok,
        "missing_columns": missing_columns if row_count else [],
        "error": error,
    })

raw_window_audit = pl.DataFrame(raw_window_rows)
expected_raw_names = {row["file"] for row in raw_window_rows}
actual_raw_paths = sorted(RAW_LOG_DIR.glob("year=*/month=*/*.parquet"))
separate_raw_files = sorted((ARCHIVE_ROOT / 'raw' / 'raw_protocol_logs').glob('year=*/month=*/*.parquet'))
unexpected_raw_files = [
    str(path.relative_to(ARCHIVE_ROOT))
    for path in actual_raw_paths if path.name not in expected_raw_names
]
raw_window_errors = raw_window_audit.filter(
    ~pl.col("status").is_in(["ok", "empty_window_review"])
)
empty_raw_windows = raw_window_audit.filter(pl.col("status") == "empty_window_review")
raw_logs_window_complete = (
    raw_window_errors.height == 0
    and not unexpected_raw_files
    and not separate_raw_files
    and not manifest_parse_errors
)
print(f"Expected consolidated protocol months: {len(raw_window_rows)} ({RAW_LOG_WINDOW_DAYS})")
display(
    raw_window_audit.group_by("status")
    .agg(pl.len().alias("windows"), pl.col("rows").sum().alias("rows"))
    .sort("status")
)
display(raw_window_audit.select([
    "window_start", "status", "rows", "bytes", "checksum_ok"
]))
print("Unexpected raw-log files:", unexpected_raw_files)
print("Unconsolidated raw-log files:", len(separate_raw_files))
print("Manifest parse errors:", manifest_parse_errors)

# Lazy Polars aggregations: inspect all logs without collecting their full contents into memory.
if raw_scan_files:
    raw_logs_lf = pl.scan_parquet(raw_scan_files, glob=False).filter(pl.col('source_table') == 'raw_protocol_logs')
    raw_log_overview = raw_logs_lf.select(
        pl.len().alias("rows"),
        pl.col("block_timestamp").min().alias("min_time"),
        pl.col("block_timestamp").max().alias("max_time"),
        pl.col("topic0").null_count().alias("null_topic0_rows"),
    ).collect().row(0, named=True)
    raw_log_contract_counts = raw_logs_lf.group_by("contract_key").agg(
        pl.len().alias("rows")
    ).collect()
    raw_log_topic_counts = raw_logs_lf.group_by(["contract_key", "topic0"]).agg(
        pl.len().alias("rows")
    ).collect()
    raw_log_unique_count = int(raw_logs_lf.select(
        pl.struct(["transaction_hash", "log_index", "contract_address"]).n_unique()
    ).collect().item())
    raw_log_duplicate_rows = int(raw_log_overview["rows"]) - raw_log_unique_count
    raw_log_out_of_scope_rows = int(raw_logs_lf.filter(
        (pl.col("block_timestamp") < START) | (pl.col("block_timestamp") >= END)
    ).select(pl.len()).collect().item())

    address_to_key = {
        REGISTRY["contracts"][key]["address"].lower(): key for key in RAW_LOG_KEYS
    }
    contract_check = raw_logs_lf.with_columns(
        pl.col("contract_address").replace_strict(address_to_key, default="UNKNOWN")
        .alias("registry_contract_key")
    )
    raw_log_contract_mismatches = int(contract_check.filter(
        pl.col("registry_contract_key") != pl.col("contract_key")
    ).select(pl.len()).collect().item())
else:
    raw_logs_lf = None
    raw_log_overview = {
        "rows": 0, "min_time": None, "max_time": None, "null_topic0_rows": 0
    }
    raw_log_contract_counts = pl.DataFrame(
        schema={"contract_key": pl.String, "rows": pl.UInt32}
    )
    raw_log_topic_counts = pl.DataFrame(
        schema={"contract_key": pl.String, "topic0": pl.String, "rows": pl.UInt32}
    )
    raw_log_duplicate_rows = 0
    raw_log_out_of_scope_rows = 0
    raw_log_contract_mismatches = 0

# Match topic0 *per contract* using the active ABI, including supplemental events.
topic_signature_rows = []
for contract_key in RAW_LOG_KEYS:
    for topic0, spec in CONTRACT_SPECS[contract_key].items():
        topic_signature_rows.append({
            "contract_key": contract_key, "topic0": topic0,
            "event_signature": spec['signature'],
        })
topic_lookup = (
    pl.DataFrame(topic_signature_rows).unique(subset=["contract_key", "topic0"])
    if topic_signature_rows
    else pl.DataFrame(schema={"contract_key": pl.String, "topic0": pl.String, "event_signature": pl.String})
)
if raw_log_topic_counts.height:
    raw_log_topic_counts = raw_log_topic_counts.join(
        topic_lookup, on=["contract_key", "topic0"], how="left"
    ).sort(["contract_key", "rows"], descending=[False, True])
    unknown_topic_counts = raw_log_topic_counts.filter(
        pl.col("event_signature").is_null()
    )
else:
    raw_log_topic_counts = raw_log_topic_counts.with_columns(
        pl.lit(None, dtype=pl.String).alias("event_signature")
    )
    unknown_topic_counts = raw_log_topic_counts

observed_contract_counts = {
    row["contract_key"]: int(row["rows"])
    for row in raw_log_contract_counts.to_dicts()
}
raw_log_contract_coverage = pl.DataFrame([
    {
        "contract_key": key,
        "protocol": REGISTRY["contracts"][key].get("protocol"),
        "rows": observed_contract_counts.get(key, 0),
    }
    for key in RAW_LOG_KEYS
]).sort(["protocol", "contract_key"])

raw_signature_rows = []
for contract_key in RAW_LOG_KEYS:
    for signature in REGISTRY["contracts"][contract_key].get("decoded_event_signatures", []):
        topic0 = "0x" + keccak(text=signature).hex()
        match = raw_log_topic_counts.filter(
            (pl.col("contract_key") == contract_key) & (pl.col("topic0") == topic0)
        )
        count = int(match["rows"].sum() or 0) if match.height else 0
        raw_signature_rows.append({
            "contract_key": contract_key, "topic0": topic0,
            "event_signature": signature, "rows": count,
        })
raw_configured_signature_coverage = pl.DataFrame(raw_signature_rows)
zero_raw_log_contracts = raw_log_contract_coverage.filter(
    pl.col("rows") == 0
)["contract_key"].to_list()
unknown_topic_rows = (
    int(unknown_topic_counts["rows"].sum() or 0) if unknown_topic_counts.height else 0
)
# The frozen archive records the historical decoder status. Re-test just those rows
# with the current ABI rather than rewriting immutable raw Parquet.
supplemental_decode_results = []
if raw_logs_lf is not None:
    formerly_unknown = raw_logs_lf.filter(pl.col('decode_status') == 'unknown_topic').select(
        ['contract_key', 'topic0', 'topics', 'data']
    ).collect().to_dicts()
    supplemental_decode_results = [
        decode_log_record(row, CONTRACT_SPECS, ABI_CONFIG['decoder_version'])
        for row in formerly_unknown
    ]
original_unknown_topic_rows = len(supplemental_decode_results)
supplementally_decoded_rows = sum(row['decode_status'] == 'decoded' for row in supplemental_decode_results)
supplemental_decode_failures = [
    {'contract_key': row['contract_key'], 'topic0': row['topic0'], 'error': row['decode_error']}
    for row in supplemental_decode_results if row['decode_status'] != 'decoded'
]
zero_signature_assessment = []
for row in missing_event_signatures.to_dicts():
    key, signature = row['contract_key'], row['event_signature']
    if key not in RAW_LOG_KEYS or not raw_logs_window_complete:
        status = 'unverified_without_complete_raw_log_fallback'
    else:
        matches = raw_configured_signature_coverage.filter(
            (pl.col('contract_key') == key) & (pl.col('event_signature') == signature)
        )
        status = ('zero_in_selected_raw_archive' if not matches.height or int(matches['rows'].sum() or 0) == 0
                  else 'raw_logs_present_but_decoded_events_missing')
    zero_signature_assessment.append({**row, 'assessment': status})
raw_logs_structurally_valid = (
    raw_logs_window_complete
    and raw_log_duplicate_rows == 0
    and raw_log_out_of_scope_rows == 0
    and raw_log_contract_mismatches == 0
    and int(raw_log_overview["null_topic0_rows"]) == 0
)
print("Raw log rows:", int(raw_log_overview["rows"]))
print("Observed time range:", raw_log_overview["min_time"], "to", raw_log_overview["max_time"])
print(
    "Duplicate log keys:", raw_log_duplicate_rows,
    "| out-of-scope timestamps:", raw_log_out_of_scope_rows,
    "| contract mapping mismatches:", raw_log_contract_mismatches,
)
print(
    "Rows with no topic0:", int(raw_log_overview["null_topic0_rows"]),
    "| unresolved topic rows under current ABI:", unknown_topic_rows,
)
print("Zero-row selected contracts:", zero_raw_log_contracts)
display(raw_log_contract_coverage)
display(raw_configured_signature_coverage)
display(raw_log_topic_counts)
if unknown_topic_rows:
    print("Unmapped topics remain in frozen raw files; do not claim full decoding.")
    display(unknown_topic_counts)
print('Formerly unknown rows:', original_unknown_topic_rows, '| decoded by current ABI:', supplementally_decoded_rows)
print('Supplemental decoding failures:', len(supplemental_decode_failures))
display(pl.DataFrame(zero_signature_assessment) if zero_signature_assessment else pl.DataFrame())

# The raw-log decoder output is already embedded in protocol_events.
decoder_path = NOTEBOOK_ROOT / 'src' / 'decode_raw_logs.py'
processor_path = NOTEBOOK_ROOT / 'src' / 'process_raw.py'
runner_path = NOTEBOOK_ROOT / 'run_process.ps1'
processor_source = processor_path.read_text(encoding='utf-8') if processor_path.exists() else ''
runner_source = runner_path.read_text(encoding='utf-8') if runner_path.exists() else ''
RAW_LOGS_DECODER_INTEGRATED = (
    decoder_path.exists()
    and 'raw_protocol_logs' in processor_source
    and 'raw_log_decode_quality_hourly' in processor_source
    and 'src/process_raw.py' in runner_source
)
raw_logs_ready_for_processing = (
    raw_logs_structurally_valid and RAW_LOGS_DECODER_INTEGRATED
    and unknown_topic_rows == 0 and not supplemental_decode_failures
)
print("Raw logs structurally valid:", bool(raw_logs_structurally_valid))
print("Raw logs integrated into processing:", RAW_LOGS_DECODER_INTEGRATED)

In [ ]:
# Small, bounded previews and null diagnostics (never loads the complete archive)
for dataset in [*DATASETS, RAW_LOG_DATASET]:
    candidates = sorted((ARCHIVE_ROOT / 'raw' / dataset).glob('year=*/month=*/*.parquet'))
    if not candidates:
        continue
    preview = pl.read_parquet(candidates[0]).head(5)
    print(f'--- {dataset}: {candidates[0].name} ---')
    print('columns:', preview.columns)
    display(preview)
    nulls = preview.null_count().transpose(include_header=True, header_name='column', column_names=['nulls'])
    display(nulls.filter(pl.col('nulls') > 0))


In [ ]:
# Derived-data checks are separate from the raw processing gate.
asset_hourly_files = sorted((ARCHIVE_ROOT / 'processed' / 'asset_hourly').glob('year=*/month=*/*.parquet'))
price_structure = {'status': 'not_generated', 'rows': 0, 'errors': []}
if asset_hourly_files:
    prices = pl.scan_parquet([str(path) for path in asset_hourly_files], glob=False).collect()
    target_assets = {key for key, meta in REGISTRY['assets'].items() if meta.get('target_asset')}
    expected_hours = int((END - START).total_seconds() // 3600)
    problems = []
    if prices.height != expected_hours * len(target_assets):
        problems.append('hourly grid row count mismatch')
    if prices.select(['asset', 'hour']).unique().height != prices.height:
        problems.append('duplicate asset-hour keys')
    per_asset = prices.group_by('asset').agg(
        pl.len().alias('hours'), pl.col('hour').min().alias('first'), pl.col('hour').max().alias('last')
    )
    if per_asset.filter((pl.col('hours') != expected_hours) |
                        (pl.col('first') != START) |
                        (pl.col('last') != END - timedelta(hours=1))).height:
        problems.append('asset-hour timeline gap or boundary mismatch')
    if set(prices['asset'].unique().to_list()) != target_assets:
        problems.append('asset universe mismatch')
    if prices.filter(pl.col('market_missing') != pl.col('market_price_usd').is_null()).height:
        problems.append('market missingness flag mismatch')
    if prices.filter(pl.col('reference_missing') != pl.col('reference_price_usd').is_null()).height:
        problems.append('reference missingness flag mismatch')
    if prices.filter(pl.col('market_price_age_hours') < 0).height:
        problems.append('future-dated market observation')
    if prices.filter((pl.col('market_price_usd') <= 0) | (pl.col('reference_price_usd') <= 0)).height:
        problems.append('nonpositive observed price or reference')
    if prices.filter((pl.col('market_price_usd').is_not_null() & ~pl.col('market_price_usd').is_finite()) |
                     (pl.col('reference_price_usd').is_not_null() & ~pl.col('reference_price_usd').is_finite())).height:
        problems.append('non-finite observed price or reference')
    if prices.filter((~pl.col('market_missing') & ~pl.col('reference_missing')) &
                     pl.col('relative_deviation').is_null()).height:
        problems.append('missing deviation despite both prices')
    if prices.filter((pl.col('eth_oracle_age_seconds') < 0) |
                     (pl.col('btc_oracle_age_seconds') < 0)).height:
        problems.append('future-dated oracle update')
    residual = (pl.col('relative_deviation') - (pl.col('market_price_usd') / pl.col('reference_price_usd') - 1)).abs()
    if prices.filter(residual > 1e-8).height:
        problems.append('deviation formula mismatch')
    price_structure = {
        'status': 'structural_checks_pass' if not problems else 'structural_checks_fail',
        'rows': prices.height, 'errors': problems,
        'market_missing_rows': int(prices['market_missing'].sum()),
        'reference_missing_rows': int(prices['reference_missing'].sum()),
        'forward_filled_rows': int(prices['market_price_forward_filled'].sum()),
        'market_sources': prices.group_by(['asset', 'market_source']).agg(pl.len().alias('rows')).to_dicts(),
    }
quality_files = sorted((ARCHIVE_ROOT / 'processed' / 'raw_log_decode_quality_hourly').glob('year=*/month=*/*.parquet'))
derived_unknown_rows = (
    int(pl.scan_parquet([str(path) for path in quality_files], glob=False).filter(
        pl.col('decode_status') != 'decoded'
    ).select(pl.col('log_count').sum()).collect().item() or 0) if quality_files else None
)
event_derivatives_current = len(quality_files) == len(MONTHS) and derived_unknown_rows == 0
labels_exist = (ARCHIVE_ROOT / 'processed' / 'labels' / 'labels.parquet').exists()
temporal_exposures_exist = bool(list((ARCHIVE_ROOT / 'tigergraph' / 'temporal').glob('*.parquet')))
print('Price structure:', price_structure['status'], '| rows:', price_structure['rows'])
print('Derived event decoding current:', event_derivatives_current, '| unresolved rows:', derived_unknown_rows)
print('Labels generated:', labels_exist, '| temporal graph exposures generated:', temporal_exposures_exist)


In [ ]:
# Final processing gate and persistent EDA report
missing_partitions = inventory.filter(pl.col('status') == 'missing')
partition_errors = inventory.filter(~pl.col('status').is_in(['ok', 'missing']))
hourly_rows = inventory.filter(pl.col('dataset') == 'hourly_blocks')['rows'].sum()
raw_backed_contracts = set(
    raw_log_contract_coverage.filter(pl.col('rows') > 0)['contract_key'].to_list()
) if raw_log_contract_coverage.height else set()
zero_event_contract_keys = zero_event_contracts['contract_key'].to_list() if 'contract_key' in zero_event_contracts.columns else []
uncovered_zero_event_contracts = [
    key for key in zero_event_contract_keys
    if key not in raw_backed_contracts
]
monthly_inputs_ready = (
    missing_partitions.height == 0
    and partition_errors.height == 0
    and not uncovered_zero_event_contracts
    and not missing_token_assets
    and partial_rpc_rows == 0
    and not zero_rpc_entities
)
ready_for_processing = monthly_inputs_ready and raw_logs_ready_for_processing
summary = {
    'schema_version': 'assetguardx-trail-eda/v4',
    'generated_at': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'archive_root': str(ARCHIVE_ROOT),
    'ready_for_processing': bool(ready_for_processing),
    'monthly_inputs_ready': bool(monthly_inputs_ready),
    'consolidated_raw_logs': {
        'partition_rule': RAW_LOG_WINDOW_DAYS,
        'expected_months': len(raw_window_rows),
        'months_complete': bool(raw_logs_window_complete),
        'structurally_valid': bool(raw_logs_structurally_valid),
        'decoder_integrated_into_processing': RAW_LOGS_DECODER_INTEGRATED,
        'ready_for_processing': bool(raw_logs_ready_for_processing),
        'rows': int(raw_log_overview['rows']),
        'min_time': raw_log_overview['min_time'],
        'max_time': raw_log_overview['max_time'],
        'duplicate_log_keys': raw_log_duplicate_rows,
        'out_of_scope_timestamp_rows': raw_log_out_of_scope_rows,
        'contract_mapping_mismatches': raw_log_contract_mismatches,
        'null_topic0_rows': int(raw_log_overview['null_topic0_rows']),
        'unknown_topic_rows': unknown_topic_rows,
        'original_unknown_topic_rows': original_unknown_topic_rows,
        'supplementally_decoded_rows': supplementally_decoded_rows,
        'supplemental_decode_failures': supplemental_decode_failures[:20],
        'active_decoder_version': ABI_CONFIG['decoder_version'],
        'derived_event_decoding_current': bool(event_derivatives_current),
        'derived_unresolved_log_rows': derived_unknown_rows,
        'zero_row_contracts': zero_raw_log_contracts,
        'unexpected_files': unexpected_raw_files,
        'unconsolidated_source_files': [str(path) for path in separate_raw_files],
        'manifest_errors': manifest_parse_errors,
        'window_errors': raw_window_errors.to_dicts(),
        'empty_windows_for_review': empty_raw_windows.to_dicts(),
        'window_audit': raw_window_audit.to_dicts(),
        'contract_coverage': raw_log_contract_coverage.to_dicts(),
        'configured_signature_coverage': raw_configured_signature_coverage.to_dicts(),
        'topic_counts': raw_log_topic_counts.to_dicts(),
        'unmapped_topic_counts': unknown_topic_counts.to_dicts(),
    },
    'missing_partitions': missing_partitions.to_dicts(),
    'partition_errors': partition_errors.to_dicts(),
    'zero_event_contracts': zero_event_contracts.to_dicts(),
    'zero_event_contracts_covered_by_raw_logs': sorted(set(zero_event_contract_keys) & raw_backed_contracts),
    'zero_event_contracts_without_fallback': uncovered_zero_event_contracts,
    'missing_event_signatures': missing_event_signatures.to_dicts(),
    'zero_signature_assessment': zero_signature_assessment,
    'research_validation': {
        'price_structure': price_structure,
        'price_economic_reconciliation': 'pending_independent_reference_comparison',
        'labels': 'generated_unvalidated' if labels_exist else 'not_generated',
        'graph_exposures': 'generated_unvalidated' if temporal_exposures_exist else 'not_generated',
        'research_ready': False,
    },
    'missing_token_assets': missing_token_assets,
    'partial_rpc_rows': int(partial_rpc_rows),
    'zero_rpc_entities': zero_rpc_entities,
    'inventory': inventory.to_dicts(),
    'event_coverage': event_coverage.to_dicts(),
    'event_signature_coverage': event_signature_coverage.to_dicts(),
    'token_coverage': token_coverage.to_dicts(),
    'rpc_coverage': rpc_coverage.to_dicts(),
    'rpc_quality': rpc_quality.to_dicts(),
    'rpc_error_examples': rpc_error_examples.to_dicts(),
}
REPORT_PATH = ARCHIVE_ROOT / 'metadata' / 'eda_raw_report.json'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(summary, indent=2, default=str) + '\n', encoding='utf-8')
print('Monthly input checks pass:', bool(monthly_inputs_ready))
print('Raw-log archive checks pass:', bool(raw_logs_structurally_valid))
print('Raw logs integrated into processor:', RAW_LOGS_DECODER_INTEGRATED)
print('READY FOR PROCESSING:', bool(ready_for_processing))
print('RESEARCH-READY: not assessed here; prices, labels, and graph exposures need separate validation')
print('EDA report:', REPORT_PATH)
if not ready_for_processing:
    print('Do not run canonical processing yet. Inspect partition, coverage, and raw-log errors in this report.')
else:
    print('Raw archive passed the EDA gate. Canonical processing can now run from Drive only.')